# Damped Harmonic Oscillator: Interactive Comparison

Explore critical, overdamped, and underdamped motion by changing the initial conditions and damping coefficients. The notebook uses `ipywidgets` for controls and ordinary Matplotlib figure output; it does **not** require the interactive Matplotlib (`ipympl`) backend.

## Before you begin
Run the cells from top to bottom. If the controls below do not appear as sliders and a button, the notebook environment may not have the Jupyter widget frontend enabled.

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

print('NumPy:', np.__version__)
print('Matplotlib:', plt.matplotlib.__version__)
print('ipywidgets:', widgets.__version__)
print('Matplotlib backend:', plt.get_backend())

NumPy: 2.4.6
Matplotlib: 3.10.9
ipywidgets: 7.8.5
Matplotlib backend: module://matplotlib_inline.backend_inline



## Model

We use the damped-oscillator equation

$$
\frac{d^2x}{dt^2} + 2\gamma\frac{dx}{dt} + \omega_0^2 x = 0
$$

For this demonstration, the natural angular frequency is fixed at $\omega_0 = 1$. The critical case therefore has $\gamma = 1$. The initial position and velocity are shared by all three curves.


In [4]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Parameters
w0 = 1.0
t = np.linspace(0.0, 12.0, 1201)

# Sliders
spos = widgets.FloatSlider(
    value=5.0, min=0.0, max=10.0, step=0.1,
    description="Initial position", continuous_update=False
)

svel = widgets.FloatSlider(
    value=0.0, min=-10.0, max=10.0, step=0.1,
    description="Initial velocity", continuous_update=False
)

sgam1 = widgets.FloatSlider(
    value=1.5, min=1.01, max=5.0, step=0.01,
    description="Gamma Over", continuous_update=False
)

sgam2 = widgets.FloatSlider(
    value=0.5, min=0.0, max=0.99, step=0.01,
    description="Gamma Under", continuous_update=False
)


# Plotting function
def plot_oscillators(pos, vel, gamma1, gamma2):

    # Critical damping: gamma = w0 = 1
    critical = (
        pos + (vel + w0 * pos) * t
    ) * np.exp(-w0 * t)

    # Overdamped case: gamma > w0
    q = np.sqrt(gamma1**2 - w0**2)

    A1 = (vel + (gamma1 + q) * pos) / (2 * q)
    A2 = ((q - gamma1) * pos - vel) / (2 * q)

    overdamped = (
        A1 * np.exp((-gamma1 + q) * t)
        + A2 * np.exp((-gamma1 - q) * t)
    )

    # Underdamped case: gamma < w0
    wd = np.sqrt(w0**2 - gamma2**2)

    underdamped = np.exp(-gamma2 * t) * (
        pos * np.cos(wd * t)
        + (vel + gamma2 * pos) / wd * np.sin(wd * t)
    )

    # Plot all three solutions
    fig, ax = plt.subplots(figsize=(9, 5))

    ax.plot(t, critical, color="blue", lw=2,
            label="Critical damping")
    ax.plot(t, overdamped, color="red", lw=2,
            label=f"Overdamped (γ = {gamma1:.2f})")
    ax.plot(t, underdamped, color="green", lw=2,
            label=f"Underdamped (γ = {gamma2:.2f})")

    ax.set_xlim(0, 12)
    ax.set_ylim(-10, 10)
    ax.set_xlabel("Time")
    ax.set_ylabel("Position")
    ax.set_title("Damped Harmonic Oscillator")
    ax.axhline(0, color="black", lw=0.7)
    ax.grid(True, alpha=0.3)
    ax.legend()

    fig.tight_layout()
    plt.show()
    plt.close(fig)


# Connect sliders to the plotting function
plot_output = widgets.interactive_output(
    plot_oscillators,
    {
        "pos": spos,
        "vel": svel,
        "gamma1": sgam1,
        "gamma2": sgam2
    }
)


# Reset button
reset_button = widgets.Button(description="Reset")

def reset_values(button):
    spos.value = 5.0
    svel.value = 0.0
    sgam1.value = 1.5
    sgam2.value = 0.5

reset_button.on_click(reset_values)


# Display controls and plot output
controls = widgets.VBox([
    spos,
    svel,
    sgam1,
    sgam2,
    reset_button
])

display(controls, plot_output)

Output()

## Questions to investigate

1. Which curve crosses equilibrium more than once?
2. How does increasing the overdamped damping coefficient change the time required to return near equilibrium?
3. How does the underdamped curve change as \\(\\gamma\\) approaches the critical value of 1?
4. Change the initial velocity while keeping the initial position fixed. Which features of the curves change?

## Technical note

The sliders are `ipywidgets` controls. Every change recalculates the curves and redraws a standard Matplotlib figure. This avoids relying on Matplotlib's interactive canvas backend, although the Anaconda Notebooks frontend must still support standard Jupyter widgets.